In [1]:
!pip install -U xgboost

In [2]:
!pip install -U lightgbm

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
import xgboost as xgb
import lightgbm as lgb
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [2]:
try:
    train = pd.read_csv('/content/train.csv')
    test = pd.read_csv('/content/test.csv')
    interactions = pd.read_csv('/content/interactions.csv')
    sample_submission = pd.read_csv('/content/sample_submission_file.csv')

except FileNotFoundError as e:
    print(f"✗ Error: {e}")
    print("\nError in uploading files")
    exit()

In [3]:
print(train.head())
print(f"\nShape: {train.shape}")
print(f"\nColumns: {train.columns.tolist()}")
print(f"\nData Types:\n{train.dtypes}")

  service_date  origin_hub_id  destination_hub_id  final_service_units
0   01-03-2023             45                  46                 2838
1   01-03-2023             46                  45                 2298
2   01-03-2023             45                  47                 2720
3   01-03-2023             47                  45                 2580
4   01-03-2023             46                   9                 4185

Shape: (67200, 4)

Columns: ['service_date', 'origin_hub_id', 'destination_hub_id', 'final_service_units']

Data Types:
service_date           object
origin_hub_id           int64
destination_hub_id      int64
final_service_units     int64
dtype: object


In [4]:
print(train['final_service_units'].describe())
print(f"Skewness: {train['final_service_units'].skew():.2f}")
print(f"Kurtosis: {train['final_service_units'].kurtosis():.2f}")

count    67200.000000
mean      2001.729464
std       1194.711140
min          2.000000
25%       1252.000000
50%       1685.000000
75%       2408.000000
max      13503.000000
Name: final_service_units, dtype: float64
Skewness: 1.91
Kurtosis: 5.76


In [5]:
print(interactions.head())
print(f"\nShape: {interactions.shape}")
print(f"\nColumns: {interactions.columns.tolist()}")

  service_date interaction_date  origin_hub_id  destination_hub_id  \
0   01-03-2023       30-01-2023             45                  46   
1   01-03-2023       30-01-2023             46                  45   
2   01-03-2023       30-01-2023             45                  47   
3   01-03-2023       30-01-2023             47                  45   
4   01-03-2023       30-01-2023             46                   9   

    origin_region destination_region origin_hub_tier destination_hub_tier  \
0       Karnataka         Tamil Nadu          Tier 1               Tier 1   
1      Tamil Nadu          Karnataka          Tier 1               Tier 1   
2       Karnataka     Andhra Pradesh          Tier 1               Tier 1   
3  Andhra Pradesh          Karnataka          Tier 1               Tier 1   
4      Tamil Nadu         Tamil Nadu          Tier 1                Tier2   

   cumulative_commitments  cumulative_interest_signals  days_before_service  
0                     8.0             

In [6]:
print(test.head())
print(f"\nShape: {test.shape}")
print(f"\nColumns: {test.columns.tolist()}")

        service_key service_date  origin_hub_id  destination_hub_id
0  2025-02-11_46_45   11-02-2025             46                  45
1  2025-01-20_17_23   20-01-2025             17                  23
2  2025-01-08_02_14   08-01-2025              2                  14
3  2025-01-08_08_47   08-01-2025              8                  47
4  2025-01-08_09_46   08-01-2025              9                  46

Shape: (5900, 4)

Columns: ['service_key', 'service_date', 'origin_hub_id', 'destination_hub_id']


In [7]:
print(train.isnull().sum())


service_date           0
origin_hub_id          0
destination_hub_id     0
final_service_units    0
dtype: int64


In [8]:
print(test.isnull().sum())

service_key           0
service_date          0
origin_hub_id         0
destination_hub_id    0
dtype: int64


In [9]:
print(interactions.isnull().sum())

service_date                   0
interaction_date               0
origin_hub_id                  0
destination_hub_id             0
origin_region                  0
destination_region             0
origin_hub_tier                1
destination_hub_tier           1
cumulative_commitments         1
cumulative_interest_signals    1
days_before_service            1
dtype: int64


In [10]:
print(f"Train service dates: {train['service_date'].min()} to {train['service_date'].max()}")
print(f"Test service dates: {test['service_date'].min()} to {test['service_date'].max()}")
print(f"Interaction dates: {interactions['interaction_date'].min()} to {interactions['interaction_date'].max()}")

Train service dates: 01-01-2024 to 31-12-2024
Test service dates: 01-01-2025 to 31-01-2025
Interaction dates: 01-02-2023 to 31-01-2023


In [11]:
train['service_date'] = pd.to_datetime(train['service_date'], format='%d-%m-%Y', dayfirst=True)
test['service_date'] = pd.to_datetime(test['service_date'], format='%d-%m-%Y', dayfirst=True)
interactions['service_date'] = pd.to_datetime(interactions['service_date'], format='%d-%m-%Y', dayfirst=True)
interactions['interaction_date'] = pd.to_datetime(interactions['interaction_date'], format='%d-%m-%Y', dayfirst=True)

In [12]:
print(f"Total interactions before filtering: {len(interactions)}")
interactions_15d = interactions[interactions['days_before_service'] >= 15].copy()
print(f"Interactions after 15-day filter: {len(interactions_15d)}")
print(f"Filtered out: {len(interactions) - len(interactions_15d)} records")

Total interactions before filtering: 54761
Interactions after 15-day filter: 28800
Filtered out: 25961 records


In [13]:
def create_aggregated_features(df):
    df_sorted = df.sort_values('days_before_service', ascending=False)
    snapshot_features = df_sorted.groupby(['service_date', 'origin_hub_id', 'destination_hub_id']).agg({
        'cumulative_commitments': 'first',
        'cumulative_interest_signals': 'first',
        'days_before_service': 'first',
        'origin_region': 'first',
        'destination_region': 'first',
        'origin_hub_tier': 'first',
        'destination_hub_tier': 'first'
    }).reset_index()
    stats_features = df.groupby(['service_date', 'origin_hub_id', 'destination_hub_id']).agg({
        'cumulative_commitments': ['max', 'min', 'mean', 'std'],
        'cumulative_interest_signals': ['max', 'min', 'mean', 'std'],
        'days_before_service': ['count', 'max', 'min']
    }).reset_index()

    stats_features.columns = ['service_date', 'origin_hub_id', 'destination_hub_id',
                               'commit_max', 'commit_min', 'commit_mean', 'commit_std',
                               'interest_max', 'interest_min', 'interest_mean', 'interest_std',
                               'interaction_count', 'days_max', 'days_min']
    features = snapshot_features.merge(stats_features,
                                       on=['service_date', 'origin_hub_id', 'destination_hub_id'],
                                       how='left')

    return features

In [14]:
features_15d = create_aggregated_features(interactions_15d)
print(f"Aggregated feature matrix created: {features_15d.shape}")

Aggregated feature matrix created: (1800, 21)


In [16]:
train_features = train.merge(features_15d,
                              on=['service_date', 'origin_hub_id', 'destination_hub_id'],
                              how='left')
test_features = test.merge(features_15d,
                            on=['service_date', 'origin_hub_id', 'destination_hub_id'],
                            how='left')
print(f"Train features: {train_features.shape}")
print(f"Test features: {test_features.shape}")

Train features: (67200, 22)
Test features: (5900, 22)


In [17]:
def extract_date_features(df):
    df['year'] = df['service_date'].dt.year
    df['month'] = df['service_date'].dt.month
    df['day'] = df['service_date'].dt.day
    df['dayofweek'] = df['service_date'].dt.dayofweek
    df['quarter'] = df['service_date'].dt.quarter
    df['week_of_year'] = df['service_date'].dt.isocalendar().week
    df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)
    df['is_month_start'] = df['service_date'].dt.is_month_start.astype(int)
    df['is_month_end'] = df['service_date'].dt.is_month_end.astype(int)
    df['days_in_month'] = df['service_date'].dt.days_in_month
    return df


In [18]:
train_features = extract_date_features(train_features)
test_features = extract_date_features(test_features)

In [19]:
le_dict = {}
categorical_cols = ['origin_hub_id', 'destination_hub_id', 'origin_region',
                    'destination_region', 'origin_hub_tier', 'destination_hub_tier']

In [20]:
for col in categorical_cols:
    if col in train_features.columns:
        le = LabelEncoder()
        combined = pd.concat([
            train_features[col].astype(str).fillna('unknown'),
            test_features[col].astype(str).fillna('unknown')
        ])
        le.fit(combined)
        train_features[col + '_encoded'] = le.transform(train_features[col].astype(str).fillna('unknown'))
        test_features[col + '_encoded'] = le.transform(test_features[col].astype(str).fillna('unknown'))
        le_dict[col] = le

In [21]:
for df in [train_features, test_features]:
    df['commit_to_interest_ratio'] = df['cumulative_commitments'] / (df['cumulative_interest_signals'] + 1)
    df['interest_to_commit_ratio'] = df['cumulative_interest_signals'] / (df['cumulative_commitments'] + 1)
    df['total_signals'] = df['cumulative_commitments'] + df['cumulative_interest_signals']
    df['commitment_rate'] = df['cumulative_commitments'] / (df['total_signals'] + 1)
    df['commit_range'] = df['commit_max'] - df['commit_min']
    df['interest_range'] = df['interest_max'] - df['interest_min']
    df['commit_growth'] = df['cumulative_commitments'] - df['commit_min']
    df['interest_growth'] = df['cumulative_interest_signals'] - df['interest_min']

In [22]:
train_features['route'] = train_features['origin_hub_id'].astype(str) + '_' + train_features['destination_hub_id'].astype(str)
test_features['route'] = test_features['origin_hub_id'].astype(str) + '_' + test_features['destination_hub_id'].astype(str)

In [23]:
route_stats = train_features.groupby('route')['final_service_units'].agg(['mean', 'median', 'std', 'min', 'max', 'count']).reset_index()
route_stats.columns = ['route', 'route_mean_demand', 'route_median_demand', 'route_std_demand',
                        'route_min_demand', 'route_max_demand', 'route_frequency']

In [24]:
train_features = train_features.merge(route_stats, on='route', how='left')
test_features = test_features.merge(route_stats, on='route', how='left')

In [25]:
origin_stats = train_features.groupby('origin_hub_id')['final_service_units'].agg(['mean', 'std']).reset_index()
origin_stats.columns = ['origin_hub_id', 'origin_mean_demand', 'origin_std_demand']

In [26]:
dest_stats = train_features.groupby('destination_hub_id')['final_service_units'].agg(['mean', 'std']).reset_index()
dest_stats.columns = ['destination_hub_id', 'dest_mean_demand', 'dest_std_demand']

In [27]:
train_features = train_features.merge(origin_stats, on='origin_hub_id', how='left')
train_features = train_features.merge(dest_stats, on='destination_hub_id', how='left')
test_features = test_features.merge(origin_stats, on='origin_hub_id', how='left')
test_features = test_features.merge(dest_stats, on='destination_hub_id', how='left')

In [28]:
feature_cols = [

    'cumulative_commitments', 'cumulative_interest_signals',
    'commit_max', 'commit_min', 'commit_mean', 'commit_std',
    'interest_max', 'interest_min', 'interest_mean', 'interest_std',
    'interaction_count', 'days_max', 'days_min',

    'year', 'month', 'day', 'dayofweek', 'quarter', 'week_of_year',
    'is_weekend', 'is_month_start', 'is_month_end', 'days_in_month',

    'commit_to_interest_ratio', 'interest_to_commit_ratio',
    'total_signals', 'commitment_rate',
    'commit_range', 'interest_range',
    'commit_growth', 'interest_growth',


    'origin_hub_id_encoded', 'destination_hub_id_encoded',
    'origin_region_encoded', 'destination_region_encoded',
    'origin_hub_tier_encoded', 'destination_hub_tier_encoded',


    'route_mean_demand', 'route_median_demand', 'route_std_demand',
    'route_min_demand', 'route_max_demand', 'route_frequency',


    'origin_mean_demand', 'origin_std_demand',
    'dest_mean_demand', 'dest_std_demand'
]

In [29]:
feature_cols = [col for col in feature_cols if col in train_features.columns]
print(f"\nSelected {len(feature_cols)} features for modeling")


Selected 47 features for modeling


In [30]:
train_features[feature_cols] = train_features[feature_cols].fillna(0)
test_features[feature_cols] = test_features[feature_cols].fillna(0)

In [31]:
train_features[feature_cols] = train_features[feature_cols].replace([np.inf, -np.inf], 0)
test_features[feature_cols] = test_features[feature_cols].replace([np.inf, -np.inf], 0)

In [32]:
X = train_features[feature_cols]
y = train_features['final_service_units']
X_test = test_features[feature_cols]

In [33]:
print(f"X_train: {X.shape}")
print(f"y_train: {y.shape}")
print(f"X_test: {X_test.shape}")

X_train: (67200, 47)
y_train: (67200,)
X_test: (5900, 47)


In [34]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")

Training set: (53760, 47)
Validation set: (13440, 47)


In [35]:
def evaluate_model(name, model, X_train, y_train, X_val, y_val):

    model.fit(X_train, y_train)
    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    train_mae = mean_absolute_error(y_train, y_train_pred)
    val_mae = mean_absolute_error(y_val, y_val_pred)
    train_r2 = r2_score(y_train, y_train_pred)
    val_r2 = r2_score(y_val, y_val_pred)
    print(f"\nTraining Metrics:")
    print(f"  RMSE: {train_rmse:.2f}")
    print(f"  MAE:  {train_mae:.2f}")
    print(f"  R²:   {train_r2:.4f}")
    print(f"\nValidation Metrics:")
    print(f"  RMSE: {val_rmse:.2f}")
    print(f"  MAE:  {val_mae:.2f}")
    print(f"  R²:   {val_r2:.4f}")
    return {
        'model': model,
        'name': name,
        'train_rmse': train_rmse,
        'val_rmse': val_rmse,
        'train_mae': train_mae,
        'val_mae': val_mae,
        'train_r2': train_r2,
        'val_r2': val_r2
    }

In [36]:
results = []

In [37]:
model = LinearRegression()
result = evaluate_model("Linear Regression", model, X_train, y_train, X_val, y_val)
results.append(result)



Training Metrics:
  RMSE: 801.26
  MAE:  549.56
  R²:   0.5492

Validation Metrics:
  RMSE: 795.75
  MAE:  551.16
  R²:   0.5602


In [38]:
model = Ridge(alpha=1.0, random_state=42)
result = evaluate_model("Ridge Regression", model, X_train, y_train, X_val, y_val)
results.append(result)


Training Metrics:
  RMSE: 801.26
  MAE:  549.56
  R²:   0.5492

Validation Metrics:
  RMSE: 795.75
  MAE:  551.16
  R²:   0.5602


In [39]:
model = Lasso(alpha=1.0, random_state=42)
result = evaluate_model("Lasso Regression", model, X_train, y_train, X_val, y_val)
results.append(result)


Training Metrics:
  RMSE: 801.36
  MAE:  549.45
  R²:   0.5491

Validation Metrics:
  RMSE: 795.87
  MAE:  551.02
  R²:   0.5601


In [40]:
model = DecisionTreeRegressor(max_depth=10, random_state=42)
result = evaluate_model("Decision Tree", model, X_train, y_train, X_val, y_val)
results.append(result)


Training Metrics:
  RMSE: 636.59
  MAE:  429.34
  R²:   0.7154

Validation Metrics:
  RMSE: 698.64
  MAE:  464.69
  R²:   0.6610


In [41]:
model = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
result = evaluate_model("Random Forest", model, X_train, y_train, X_val, y_val)
results.append(result)


Training Metrics:
  RMSE: 355.82
  MAE:  244.64
  R²:   0.9111

Validation Metrics:
  RMSE: 537.60
  MAE:  353.82
  R²:   0.7993


In [42]:
model = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
result = evaluate_model("Gradient Boosting", model, X_train, y_train, X_val, y_val)
results.append(result)


Training Metrics:
  RMSE: 605.22
  MAE:  396.44
  R²:   0.7428

Validation Metrics:
  RMSE: 613.20
  MAE:  407.53
  R²:   0.7389


In [43]:
model = xgb.XGBRegressor(n_estimators=200, max_depth=7, learning_rate=0.1,
                         random_state=42, n_jobs=-1)
result = evaluate_model("XGBoost", model, X_train, y_train, X_val, y_val)
results.append(result)


Training Metrics:
  RMSE: 414.44
  MAE:  273.38
  R²:   0.8794

Validation Metrics:
  RMSE: 468.81
  MAE:  310.80
  R²:   0.8474


In [44]:
model = lgb.LGBMRegressor(n_estimators=200, max_depth=7, learning_rate=0.1,
                          random_state=42, n_jobs=-1, verbose=-1)
result = evaluate_model("LightGBM", model, X_train, y_train, X_val, y_val)
results.append(result)


Training Metrics:
  RMSE: 513.96
  MAE:  336.01
  R²:   0.8145

Validation Metrics:
  RMSE: 533.91
  MAE:  353.05
  R²:   0.8020


In [45]:
comparison_df = pd.DataFrame(results)
comparison_df = comparison_df[['name', 'train_rmse', 'val_rmse', 'train_mae', 'val_mae', 'train_r2', 'val_r2']]
comparison_df = comparison_df.sort_values('val_rmse')

In [46]:
best_model_result = comparison_df.iloc[0]

In [47]:
best_model_name = best_model_result['name']
best_model = [r for r in results if r['name'] == best_model_name][0]['model']

In [48]:
best_model.fit(X, y)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=7,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=200,
             n_jobs=-1, num_parallel_tree=None, ...)

In [49]:
predictions = best_model.predict(X_test)
predictions = np.maximum(predictions, 0)  # Ensure non-negative predictions
predictions = predictions.astype(int)

In [50]:
submission = test_features[['service_key']].copy()
submission['final_service_units'] = predictions

In [51]:
print(submission['final_service_units'].describe())

count    5900.000000
mean     1991.730847
std      1023.818253
min         0.000000
25%      1355.000000
50%      1721.000000
75%      2299.250000
max      8471.000000
Name: final_service_units, dtype: float64


In [52]:
output_file = 'submission.csv'
submission.to_csv(output_file, index=False)